## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


We achieve reproducibility by running all analysis tools in strictly versioned, isolated containers (Docker/Singularity) via nf-core,  together with pinned pipeline versions and documented parameters, so that results can be reproduced across different platforms and over time.

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so? 

The next step is quality control and processing of the reads, which generates the gene expression count table. This is done with the pipeline nf-core/rnaseq (FastQC, trimming, alignment, quantification, MultiQC report).

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do? 
nf-core/rnaseq performs quality control (preprocessing) of the raw reads, adapter and quality trimming, alignment to the reference genome, gene level quantification and summarises all QC metrics in a MultiQC report.

Which are the main tools that will be used in the pipeline? 
FastQC (read quality), fastp (trimming), STAR (default, HISAT2 optional) for alignment, Salmon for quantification, and MultiQC for the report.

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [2]:
import pandas as pd

df = pd.read_excel("conditions_runs_oxy_project.xlsx")

def pick(row, mapping):
    for col, label in mapping.items():
        if str(row[col]).strip().lower() == "x":
            return label

cond_map = {"condition: Sal": "Sal", "Condition: Oxy": "Oxy"}
geno_map = {"Genotype: SNI": "SNI", "Genotype: Sham": "Sham"}

tidy = pd.DataFrame({
    "run": df["Run"],
    "condition": df.apply(pick, axis=1, mapping=cond_map),
    "genotype": df.apply(pick, axis=1, mapping=geno_map),
})

basecounts_df = pd.read_csv("base_counts.csv")
tidy_bases = pd.merge(tidy, basecounts_df, left_on="run", right_on="Run", how="left").drop(columns="Run")

tidy_sorted = tidy_bases.sort_values(by=["Bases", "condition"], ascending=True)
smallest_runs = tidy_sorted.head(2)

display(smallest_runs)
smallest_runs[["run"]].to_csv("ids.csv", index=False, header=False)

,run,condition,genotype,Bases
11,SRR23195516,Oxy,SNI,6203117700
6,SRR23195511,Oxy,Sham,6456390900


In [4]:
import pandas as pd

fetch = pd.read_csv("../fetchngs_out/samplesheet/samplesheet.csv")
fetch = fetch[["run_accession", "fastq_1", "fastq_2"]]

meta = smallest_runs[["run", "genotype", "condition"]]   
merged = meta.merge(fetch, left_on="run", right_on="run_accession")

samplesheet_RNAseq = pd.DataFrame({
    "sample": merged["genotype"] + "_" + merged["condition"] + "_" + merged["run"],
    "fastq_1": merged["fastq_1"],
    "fastq_2": merged["fastq_2"],
    "strandedness": "auto",
})
samplesheet_RNAseq.to_csv("samplesheet_RNAseq.csv", index=False)
print(samplesheet_RNAseq)

                 sample                                            fastq_1  \
0   SNI_Oxy_SRR23195516  fetchngs_out/fastq/SRX19144486_SRR23195516_1.f...   
1  Sham_Oxy_SRR23195511  fetchngs_out/fastq/SRX19144488_SRR23195511_1.f...   

                                             fastq_2 strandedness  
0  fetchngs_out/fastq/SRX19144486_SRR23195516_2.f...         auto  
1  fetchngs_out/fastq/SRX19144488_SRR23195511_2.f...         auto  


In [ ]:
# post here the command you used to run nf-core/rnaseq
!nextflow run nf-core/rnaseq -r 3.27.0 -profile docker \
  --input day_02/samplesheet_RNAseq.csv \
  --outdir day_02/rnaseq_out \
  --genome GRCm38 \
  --aligner hisat2

Explain all the parameters you set and why you set them in this way.

-r 3.27.0: fixes the pipeline version, so the run is reproducible
-profile docker: runs every tool in a versioned Docker container, so the software is identical on every system
--input day_02/samplesheet_RNAseq.csv: samplesheet with sample names (genotype, condition, run ID) and the paths to the paired FASTQ files, strandedness is set to auto, so the pipeline infers it itself
--outdir day_02/rnaseq_out: folder for all results
--genome GRCm38: uses the iGenomes mouse reference (genome and annotation)
--aligner hisat2: uses HISAT2 instead of the default STAR, because it needs much less memory and time



## Browsing the results

How did the pipeline perform?

The pipeline did not complete. The samplesheet passed validation and the run started, but the process FQ_LINT failed for sample Sham_Oxy_SRR23195511 with exit status 125. The error message docker: error getting credentials shows that Docker could not pull the container image quay.io/biocontainers/fq:0.12.0--h9ee0642_0, so the failure was caused by the Docker setup and not by the pipeline, the samplesheet or the parameters. Nextflow then cancelled the run. Because of time constraints, I proceeded using the provided nf-core/rnaseq output for the next tasks instead.

Explain the quality control steps. Are you happy with the quality and why. If not, why not.

Quality control in nf-core/rnaseq runs at several levels. FastQC checks the raw reads (base quality, adapter content, GC content, duplication). Trimming (Trim Galore or fastp) removes adapters and low-quality bases, followed by a second FastQC run to verify the trimming. After alignment (using STAR), SAMtools and Qualimap report the mapping rate and the distribution of reads over exons, introns and intergenic regions. RSeQC checks strandedness and gene body coverage. Picard MarkDuplicates and dupRadar assess duplication. All metrics are collected in one MultiQC report.

I am only moderately satisfied with the overall data quality. The mapping rate for most samples is high (between 82% up to 94.6%), but there are also four samples with notably lower alignment rates (73.6%, 65.2%, 59.5%, and 28.2%). The duplication rates are quite mixed and reach up to 88%, while the remaining samples stay between ~23 % and 32 %, indicating variable library complexity across the dataset. The base sequence quality is consistently high across all samples. The mean Phred scores remain well above the quality threshold (mostly >30) along the entire length of the reads, indicating reliable sequencing performance with minimal error rates. The sequencing depth is relatively consistent across samples (approx. 20–27M reads per sample). However, the proportion of unique vs. duplicate reads varies: while high-quality samples (e.g., SRX19144485) consist mostly of unique reads, outlier samples like SRX19144489 show strong PCR duplication with only a small fraction of unique reads remaining.
Regarding the GC content: Most samples display a smooth, unimodal distribution centered around 45–50% GC. However, four outlier samples exhibit abnormal secondary peaks in high GC regions (up to ~85% GC). This may be related with the high duplication rates and poor mapping efficiency observed earlier, possibly indicating over-represented sequences or contamination.


Please give additional information on : 
- ribosomal rRNA: 
rRNA reads are uninformative and reduce the effective sequencing depth. If the rRNA proportion is high, I would remove it or consider rRNA depletion in the library preparation.
- Duplication: 
A high duplication rate means that a small number of fragments are sequenced repeatedly, so fewer distinct genes are covered. In RNA-seq, part of it is natural (highly expressed genes), which is why duplicates are flagged but generally not removed.
- GC content contamination parts:  
An abnormal GC profile can indicate contamination or sequence bias, which would distort normalisation and expression estimates. I would identify the source with a contamination screen before deciding what to exclude.

What are the possible steps that could lead to poorer results?

Low RNA quality or degradation, adapter contamination, high rRNA content, strong PCR duplication or low library complexity, contamination from other species, low sequencing depth, wrong strandedness setting, a mismatched reference genome or annotation, and batch effects.

Would you exclude any samples? If yes, which and why?

Yes, I would consider excluding the outlier samples SRX19144489, SRX19144492, and SRX19144491, due to their results regarding PCR duplication rates, secondary GC peaks, and mapping rates. These samples could bias downstream differential expression analysis.

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

I would build the count table from the pipeline output, check sample metadata and PCA, and run differential expression analysis (DESeq2) for the contrasts SNI vs. Sham and Oxy vs. Saline, including the interaction. Afterwards I would do pathway or enrichment analysis and compare the results with the paper.

The scientists want to understand how oxycodone withdrawal changes gene expression in the brain reward circuitry, with and without chronic neuropathic pain, and to identify regulators such as HDAC1/HDAC2 that could be therapeutic targets.

In R, I would use DESeq2 for normalisation and differential expression, clusterProfiler for enrichment analysis, and ggplot2, pheatmap and EnhancedVolcano for PCA, heatmaps and volcano plots. In Python, the equivalent would be PyDESeq2, pandas and matplotlib/seaborn.